# **GOLD Layer - (Star Schema)**

```
silver_lakehouse (clean data)
        ↓
gold_lakehouse/
  dim_date
  dim_customer
  dim_geography
  dim_payment
  fact_orders
  agg_daily_sales
  
```

In [ ]:
# CELL 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql import types as T
from delta.tables import DeltaTable
import pandas as pd
from datetime import date, timedelta

print("Libraries loaded")

StatementMeta(, 2d48fcd9-966f-4a17-973d-841593445445, 19, Finished, Available, Finished, False)

Libraries loaded


In [ ]:
# ==========================================
# CELL 2 — Build Date Dimension (dim_date)
# Source : Generated Calendar (No Silver Table)
# Target : Gold Layer (dim_date)
# ==========================================

# NOTE:
# dim_date is generated using Spark SQL sequence().
# It does NOT read from any Silver Delta table.
# Therefore, no SILVER_DATE_PATH is required.

spark.sql("""

CREATE OR REPLACE TABLE dim_date
USING DELTA
AS

WITH dates AS (
    SELECT explode(
        sequence(
            to_date('2020-01-01'),
            to_date('2027-12-31'),
            interval 1 day
        )
    ) AS full_date
)

SELECT
    CAST(date_format(full_date, 'yyyyMMdd') AS INT) AS date_key,
    full_date,
    YEAR(full_date) AS year,
    QUARTER(full_date) AS quarter,
    MONTH(full_date) AS month_num,
    date_format(full_date, 'MMMM') AS month_name,
    date_format(full_date, 'MMM') AS month_short,
    WEEKOFYEAR(full_date) AS week_of_year,
    DAY(full_date) AS day_of_month,
    DAYOFWEEK(full_date) AS day_of_week,
    date_format(full_date, 'EEEE') AS day_name,
    CASE WHEN DAYOFWEEK(full_date) IN (1,7) THEN TRUE ELSE FALSE END AS is_weekend,
    CASE WHEN DAY(full_date) = 1 THEN TRUE ELSE FALSE END AS is_month_start,
    CASE
        WHEN MONTH(full_date) >= 4 THEN YEAR(full_date)
        ELSE YEAR(full_date) - 1
    END AS fiscal_year,
    CASE
        WHEN MONTH(full_date) BETWEEN 4 AND 6 THEN 1
        WHEN MONTH(full_date) BETWEEN 7 AND 9 THEN 2
        WHEN MONTH(full_date) BETWEEN 10 AND 12 THEN 3
        ELSE 4
    END AS fiscal_quarter,
    CAST(date_format(full_date, 'yyyyMM') AS INT) AS year_month,
    current_timestamp() AS gold_created_at

FROM dates

""")

date_count = spark.sql("""
SELECT COUNT(*) AS total_dates
FROM dim_date
""").collect()[0][0]

print(f"✓ dim_date created successfully : {date_count} rows")

StatementMeta(, 2d48fcd9-966f-4a17-973d-841593445445, 20, Finished, Available, Finished, False)

✓ dim_date created successfully : 2922 rows


In [ ]:
# CELL 3 — Build dim_customer

SILVER_CUSTOMER_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "silver_lakehouse.Lakehouse/Tables/dbo/silver_customers"
)

# abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Tables/dbo/silver_customers

# FIX: Ensure you are using the 2-part name: gold_lakehouse.dim_customer
# Do NOT use gold_lakehouse.gold_lakehouse.dim_customer
spark.sql(f"""
    CREATE OR REPLACE TABLE dim_customer
    USING DELTA AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY customer_id)  AS customer_key,
        customer_id,
        first_name,
        last_name,
        CONCAT(first_name, ' ', last_name)         AS full_name,
        city,
        state,
        loyalty_tier,
        date_joined,
        days_as_customer,
        total_orders,
        is_active,
        CASE
            WHEN days_as_customer > 730 THEN 'Long-term'
            WHEN days_as_customer > 180 THEN 'Established'
            WHEN days_as_customer > 30  THEN 'Growing'
            ELSE 'New'
        END AS tenure_segment,
        current_timestamp() AS gold_created_at
    FROM delta.`{SILVER_CUSTOMER_PATH}`
    WHERE customer_id IS NOT NULL
""")

# FIX: Use the exact same 2-part name here to verify the count
print("dim_customer:", spark.sql("SELECT COUNT(*) FROM dim_customer").collect()[0][0])

StatementMeta(, 2d48fcd9-966f-4a17-973d-841593445445, 21, Finished, Available, Finished, False)

dim_customer: 500


In [ ]:
# CELL 4 — Build dim_geography

# Note: The variable name is SILVER_CUSTOMER_PATH but it points to silver_orders. 
# This is perfectly fine for extracting distinct cities/states, but be aware of the naming.
SILVER_CUSTOMER_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "silver_lakehouse.Lakehouse/Tables/dbo/silver_orders"
)

# 1. Changed to f-string (f""")
# 2. Used delta.`{SILVER_CUSTOMER_PATH}`
# 3. Added alias "AS distinct_locations" to the subquery
# 4. Added gold_lakehouse. to the CREATE TABLE statement
spark.sql(f"""
    CREATE OR REPLACE TABLE dim_geography
    USING DELTA AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY city, state) AS geography_key,
        city,
        state,
        CASE state
            WHEN 'MH' THEN 'Maharashtra'
            WHEN 'DL' THEN 'Delhi'
            WHEN 'KA' THEN 'Karnataka'
            WHEN 'TS' THEN 'Telangana'
            WHEN 'TN' THEN 'Tamil Nadu'
            WHEN 'GJ' THEN 'Gujarat'
            WHEN 'RJ' THEN 'Rajasthan'
            WHEN 'WB' THEN 'West Bengal'
            ELSE 'Other'
        END AS state_name,
        CASE state
            WHEN 'MH' THEN 'West'
            WHEN 'DL' THEN 'North'
            WHEN 'KA' THEN 'South'
            WHEN 'TS' THEN 'South'
            WHEN 'TN' THEN 'South'
            WHEN 'GJ' THEN 'West'
            WHEN 'RJ' THEN 'North'
            WHEN 'WB' THEN 'East'
            ELSE 'Other'
        END AS region,
        CASE city
            WHEN 'Mumbai'    THEN 'Tier 1'
            WHEN 'Delhi'     THEN 'Tier 1'
            WHEN 'Bangalore' THEN 'Tier 1'
            WHEN 'Hyderabad' THEN 'Tier 1'
            WHEN 'Chennai'   THEN 'Tier 1'
            WHEN 'Kolkata'   THEN 'Tier 1'
            ELSE 'Tier 2'
        END AS city_tier
    FROM (
        SELECT DISTINCT city, state
        FROM delta.`{SILVER_CUSTOMER_PATH}`
        WHERE city IS NOT NULL AND city != 'Unknown'
    ) AS distinct_locations
""")

# 5. Added gold_lakehouse. to the SELECT statement for consistency
print("dim_geography:", spark.sql("SELECT COUNT(*) FROM dim_geography").collect()[0][0])

StatementMeta(, 2d48fcd9-966f-4a17-973d-841593445445, 22, Finished, Available, Finished, False)

dim_geography: 60


In [ ]:
# CELL 5 — Build dim_payment
SILVER_CUSTOMER_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "silver_lakehouse.Lakehouse/Tables/dbo/silver_orders"
)
spark.sql(f"""
    CREATE OR REPLACE TABLE dim_payment
    USING DELTA AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY payment_method) AS payment_key,
        payment_method,
        CASE payment_method
            WHEN 'UPI'              THEN 'Digital'
            WHEN 'Credit Card'      THEN 'Card'
            WHEN 'Debit Card'       THEN 'Card'
            WHEN 'Net Banking'      THEN 'Digital'
            WHEN 'Wallet'           THEN 'Digital'
            WHEN 'Cash on Delivery' THEN 'Cash'
            ELSE 'Other'
        END AS payment_category,
        payment_method IN ('UPI','Credit Card','Debit Card',
                           'Net Banking','Wallet') AS is_digital,
        payment_method = 'Cash on Delivery'        AS is_cod
    FROM (
        SELECT DISTINCT payment_method
        FROM delta.`{SILVER_CUSTOMER_PATH}`
        WHERE payment_method IS NOT NULL
    )
""")

print("dim_payment:", spark.sql("SELECT COUNT(*) FROM dim_payment").collect()[0][0])

StatementMeta(, 2d48fcd9-966f-4a17-973d-841593445445, 23, Finished, Available, Finished, False)

dim_payment: 6


In [ ]:
# CELL 6 — Build fact_orders (most important table)
SILVER_CUSTOMER_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "silver_lakehouse.Lakehouse/Tables/dbo/silver_orders"
)

spark.sql(f"""
    CREATE OR REPLACE TABLE fact_orders
    USING DELTA
    PARTITIONED BY (date_key)
    AS
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY so.order_id) AS BIGINT) AS order_key,

        -- Foreign keys to dimensions
        dd.date_key,
        dc.customer_key,
        dg.geography_key,
        dp.payment_key,

        -- Business keys (keep for tracing)
        so.order_id,
        so.customer_id,

        -- Measures
        so.order_amount                                    AS gross_revenue,
        so.item_count,

        -- Derived measures
        CASE WHEN so.order_status = 'returned'
             THEN so.order_amount ELSE 0 END               AS returned_amount,
        CASE WHEN so.order_status = 'cancelled'
             THEN 1 ELSE 0 END                             AS is_cancelled,
        CASE WHEN so.order_status = 'delivered'
             THEN so.order_amount ELSE 0 END               AS net_revenue,
        CASE WHEN so.discount_code != 'NO_PROMO'
             THEN 1 ELSE 0 END                             AS has_discount,

        -- Attributes
        so.order_status,
        so.is_weekend,
        so.is_guest_order,
        so.order_hour,

        CURRENT_TIMESTAMP()                                AS gold_created_at

    FROM delta.`{SILVER_CUSTOMER_PATH}` so

    -- Join date dimension
    JOIN dim_date dd
      ON dd.date_key = CAST(DATE_FORMAT(so.order_date, 'yyyyMMdd') AS INT)

    -- Join customer dimension
    LEFT JOIN dim_customer dc
      ON dc.customer_id = so.customer_id

    -- Join geography dimension
    LEFT JOIN dim_geography dg
      ON dg.city = so.city AND dg.state = so.state

    -- Join payment dimension
    LEFT JOIN dim_payment dp
      ON dp.payment_method = so.payment_method

    WHERE so.order_id IS NOT NULL
      AND so.order_amount IS NOT NULL
""")

count = spark.sql("SELECT COUNT(*) FROM fact_orders").collect()[0][0]
print("fact_orders:", count, "rows")

StatementMeta(, 2d48fcd9-966f-4a17-973d-841593445445, 24, Finished, Available, Finished, False)

fact_orders: 1400 rows


In [ ]:
# ==========================================
# CELL 7 — Build agg_daily_sales
# Pre-aggregated table for Power BI dashboards
# ==========================================

spark.sql("""

CREATE OR REPLACE TABLE agg_daily_sales
USING DELTA
AS

SELECT

    dd.full_date,
    dd.year,
    dd.quarter,
    dd.month_num,
    dd.month_name,
    dd.day_name,
    dd.is_weekend,

    dg.city,
    dg.state_name,
    dg.region,
    dg.city_tier,

    -- Revenue Metrics
    SUM(fo.gross_revenue) AS total_revenue,
    SUM(fo.net_revenue) AS net_revenue,
    SUM(fo.returned_amount) AS total_returns,

    -- Order Metrics
    COUNT(fo.order_key) AS total_orders,

    -- Total Quantity Sold
    SUM(
        aggregate(
            fo.item_count,
            CAST(0 AS BIGINT),
            (acc, item) -> acc + item.quantity
        )
    ) AS total_items,

    -- Cancelled Orders
    SUM(CAST(fo.is_cancelled AS INT)) AS cancelled_orders,

    -- Average Order Value
    ROUND(AVG(fo.gross_revenue), 2) AS avg_order_value,

    -- Cancellation Rate
    ROUND(
        SUM(CAST(fo.is_cancelled AS INT)) * 100.0 / COUNT(*),
        2
    ) AS cancellation_rate,

    -- Discount Rate
    ROUND(
        SUM(CAST(fo.has_discount AS INT)) * 100.0 / COUNT(*),
        2
    ) AS discount_rate

FROM fact_orders fo

JOIN dim_date dd
    ON fo.date_key = dd.date_key

JOIN dim_geography dg
    ON fo.geography_key = dg.geography_key

GROUP BY

    dd.full_date,
    dd.year,
    dd.quarter,
    dd.month_num,
    dd.month_name,
    dd.day_name,
    dd.is_weekend,

    dg.city,
    dg.state_name,
    dg.region,
    dg.city_tier

""")

print(
    "agg_daily_sales:",
    spark.sql("SELECT COUNT(*) FROM agg_daily_sales").collect()[0][0]
)

StatementMeta(, 2d48fcd9-966f-4a17-973d-841593445445, 25, Finished, Available, Finished, False)

agg_daily_sales: 980


In [ ]:
# CELL 8 — Final verification of all Gold tables

tables = [
    "dim_date",
    "dim_customer",
    "dim_geography",
    "dim_payment",
    "fact_orders",
    "agg_daily_sales"
]

print("=" * 45)
print("GOLD LAYER SUMMARY")
print("=" * 45)

for t in tables:
    c = spark.sql(f"SELECT COUNT(*) FROM {t}").collect()[0][0]
    print(f"{t:<25} {c:>6} rows")

print("=" * 45)
print("Gold layer complete. Ready for Phase 5.")

StatementMeta(, 2d48fcd9-966f-4a17-973d-841593445445, 26, Finished, Available, Finished, False)

GOLD LAYER SUMMARY
dim_date                    2922 rows
dim_customer                 500 rows
dim_geography                 60 rows
dim_payment                    6 rows
fact_orders                 1400 rows
agg_daily_sales              980 rows
Gold layer complete. Ready for Phase 5.
